In [2]:
pip install emoji

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install autocorrect

Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd
import re
import emoji
import string
import nltk
from bs4 import BeautifulSoup
from autocorrect import Speller
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag

# INITIALIZATION & DOWNLOADS 
# Download required NLTK resources 
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

# Initialize tools 
spell = Speller(lang='en')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Slang Dictionary 
slang_dict = {
    "tbh": "to be honest", 
    "omg": "oh my god", 
    "lol": "laugh out loud",
    "idk": "I don't know", 
    "brb": "be right back", 
    "btw": "by the way",
    "imo": "in my opinion", 
    "smh": "shaking my head", 
    "fyi": "for your information",
    "np": "no problem", 
    "ikr": "I know right", 
    "asap": "as soon as possible",
    "bff": "best friend forever", 
    "gg": "good game", 
    "hmu": "hit me up",
    "rofl": "rolling on the floor laughing"
}

# Contractions Dictionary
contractions_dict = {
    "wasn't": "was not", 
    "isn't": "is not", 
    "aren't": "are not", 
    "don't": "do not",
    "can't": "cannot", 
    "i'm": "i am", 
    "you're": "you are", 
    "it's": "it is",
    "we're": "we are", 
    "they're": "they are", 
    "i've": "i have", 
    "let's": "let us",
    "that's": "that is", 
    "won't": "will not", 
    "didn't": "did not" 
}

# --- PREPROCESSING FUNCTIONS ---

def remove_urls(text):
    """Removes URLs starting with http or www """
    return re.sub(r'http\S+|www\S+', '', text)

def remove_html(text):
    """Removes HTML tags using BeautifulSoup"""
    return BeautifulSoup(text, "html.parser").get_text()

def remove_emojis(text):
    """Replaces emojis with an empty string """
    return emoji.replace_emoji(text, replace="")

def replace_slang(text):
    """Replaces internet slang using regex """
    escaped_slang = [re.escape(word) for word in slang_dict.keys()]
    pattern = r'\b(' + '|'.join(escaped_slang) + r')\b'
    return re.sub(pattern, lambda m: slang_dict[m.group(0).lower()], text, flags=re.IGNORECASE)

def replace_contractions(text):
    """Expands contractions like 'can't' to 'cannot' """
    escaped = [re.escape(c) for c in contractions_dict.keys()]
    pattern = re.compile(r'\b(' + '|'.join(escaped) + r')\b', flags=re.IGNORECASE)
    return pattern.sub(lambda m: contractions_dict[m.group(0).lower()], text)

def remove_punctuation(text):
    """Removes all punctuation marks """
    return text.translate(str.maketrans("", "", string.punctuation))

def remove_numbers(text):
    """Removes all numeric digits """
    return re.sub(r'\d+', '', text)

def get_wordnet_pos(nltk_tag):
    """Maps NLTK POS tags to WordNet POS tags for lemmatization """
    if nltk_tag.startswith('J'): return wordnet.ADJ
    elif nltk_tag.startswith('V'): return wordnet.VERB
    elif nltk_tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

def lemmatize_text(text):
    """Lemmatizes text based on Part-of-Speech tags """
    if not isinstance(text, str): return ""
    tokens = word_tokenize(text)
    pos_tags = pos_tag(tokens)
    return " ".join([lemmatizer.lemmatize(w, get_wordnet_pos(t)) for w, t in pos_tags])

# --- PIPELINE ---

def preprocess_text(text):
    """The complete 12-step preprocessing pipeline"""
    if not isinstance(text, str): return []
    text = text.lower() # Step 1 
    text = remove_urls(text) # Step 2
    text = remove_html(text) # Step 3
    text = remove_emojis(text) # Step 4
    text = replace_slang(text) # Step 5
    text = replace_contractions(text) # Step 6
    text = remove_punctuation(text) # Step 7
    text = remove_numbers(text) # Step 8
    text = spell(text) # Step 9: Spelling correction
    # Step 10: Stopwords 
    text = " ".join([w for w in text.split() if w.lower() not in stop_words])
    text = lemmatize_text(text) # Step 11
    return word_tokenize(text) # Step 12: Tokenization 

# --- EXECUTION ---

# 1. Load data 
df = pd.read_csv("Review.csv")

# 2. Apply the pipeline 
df["processed"] = df["Review"].apply(preprocess_text)

# 3. Save and Display 
df.to_csv("Processed_Reviews_Final.csv", index=False)
pd.set_option('display.max_colwidth', None)
print(df[["Review", "processed"]].head())

                                                                          Review  \
0  The product arrived on time. Packaging was great, and the quality is amazing!   
1                                       THIS PRODUCT IS JUST AMAZING! I LOVE IT.   
2    I bought this phone for $799, and it has a 120Hz display. Totally worth it!   
3                        Wow!!! This product is awesome... but a bit expensive??   
4                                            The laptop works perfectly fine.      

                                                     processed  
0  [product, arrive, time, packaging, great, quality, amazing]  
1                                       [product, amaze, love]  
2                    [buy, phone, hz, display, totally, worth]  
3                      [wow, product, awesome, bit, expensive]  
4                              [laptop, work, perfectly, fine]  


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nabie\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\nabie\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\nabie\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\nabie\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\nabie\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
